# Streaming ML Framework — Demo Notebook

## Framework 簡介

本 Framework 是一套**純 NumPy 實作**的串流機器學習框架，專為資料以「分批（chunk）」形式陸續抵達的場景設計。
框架完全不依賴 scikit-learn、scipy 等第三方 ML 函式庫，僅使用 `numpy` 與 `matplotlib`。

### 主要模組

| 模組 | 功能 |
|---|---|
| `io.py` | 自訂 CSV 讀寫、串流生成器、資料切分 |
| `preprocessing.py` | `StandardScaler`、`MinMaxScaler`、`Imputer`、`OneHotEncoder`，全部支援 `partial_fit` |
| `stats.py` | 串流統計（`StreamStats`、`chunk_mean/variance/quantile/histogram`） |
| `tree.py` | 決策樹分類器（支援 Gini / Entropy、`partial_fit` 增量學習） |
| `ensemble.py` | `EnsembleClassifier`（Bagging / Random Forest）、`partial_fit` 串流支援 |
| `metrics.py` | 串流指標（`Accuracy`、`Precision`、`Recall`、`F1Score`、`ConfusionMatrix`、`AUC`）及批次版函數 |
| `pipeline.py` | `Pipeline`：將 transformer + estimator 串聯，支援 `partial_fit` 串流訓練 |
| `stream.py` | `StreamTrainer`：自動迭代 chunk、記錄指標、追蹤記憶體用量 |
| `visualise.py` | 繪圖工具（指標趨勢圖、模型比較圖、預測散點圖、混淆矩陣） |

### 本 Demo 涵蓋的四個核心要求

1. **使用 `io.py` 從 CSV 載入資料集**
2. **將資料集分割成多個 chunk，模擬串流資料情境**
3. **對每個 chunk 呼叫 `.partial_fit()` 進行增量訓練**
4. **使用 `visualise.py` 記錄並視覺化關鍵指標（accuracy、error rate、模型比較）**

## Cell 1 — 載入套件

匯入本 Framework 的各模組，以及 `numpy` 與 `matplotlib`。
確認 repo 根目錄已加入 `sys.path`，使 `demo/` 資料夾內也能正確 import。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

# Framework modules
from framework.io import load_csv, save_csv, split_into_chunks
from framework.preprocessing import StandardScaler, Imputer
from framework.ensemble import RandomForestClassifier, EnsembleClassifier
from framework.pipeline import Pipeline
from framework.stream import StreamTrainer
from framework.metrics import (
    Accuracy, F1Score,
    accuracy_score, f1_score,
    confusion_matrix as compute_cm,
)
from framework.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
    plot_confusion_matrix,
)

print('Imports OK')

## Cell 2 — 產生合成資料集並儲存為 CSV

使用 NumPy 隨機產生 1000 筆、6 個特徵的二元分類資料集，
標籤由前三個特徵的加權和決定（線性可分邊界）。

資料以 `save_csv`（`io.py`）寫入 `/tmp/demo_data.csv`，
模擬真實世界中需從檔案讀取資料的情境。

In [ ]:
rng = np.random.default_rng(seed=0)
N, D = 1000, 6
X_raw = rng.normal(loc=0.0, scale=2.0, size=(N, D))

# 標籤由加權線性組合決定（前 3 個特徵有效，後 3 個為雜訊）
weights = np.array([1.5, -1.0, 0.8, 0.0, 0.0, 0.0])
y_raw = (X_raw @ weights > 0).astype(int)

# 整合成單一矩陣並儲存
csv_path = '/tmp/demo_data.csv'
headers = [f'f{i}' for i in range(D)] + ['label']
save_csv(csv_path, np.column_stack([X_raw, y_raw]), headers=headers)

print(f'已儲存 {N} 筆資料至 {csv_path}')
print(f'欄位: {headers}')
print(f'類別分布: {np.bincount(y_raw)} (0: {np.sum(y_raw==0)}, 1: {np.sum(y_raw==1)})')

## Cell 3 — 使用 `io.load_csv` 載入資料集（核心要求 1）

以 `io.py` 的 `load_csv` 函數讀取 CSV 檔案，回傳 NumPy 陣列與欄位名稱清單。
空值欄位會自動轉換為 `np.nan`，無需額外處理。

In [ ]:
# 核心要求 1：使用自訂 io.py 從 CSV 載入資料
data, cols = load_csv(csv_path, has_header=True)

X = data[:, :-1]          # 特徵矩陣 (1000, 6)
y = data[:, -1].astype(int)  # 標籤向量 (1000,)

print(f'載入完成')
print(f'  資料形狀 : {data.shape}')
print(f'  欄位名稱 : {cols}')
print(f'  X shape  : {X.shape}')
print(f'  y shape  : {y.shape}')
print(f'  類別分布 : {np.bincount(y)}')

## Cell 4 — 切分資料為多個 Chunk，模擬串流情境（核心要求 2）

以 `split_into_chunks`（`io.py`）將資料集切成 10 個連續且等大的 chunk，
模擬資料以時間序列方式逐批到達的串流情境。
實際應用中，chunk 可來自感測器資料、網路日誌或即時資料庫查詢。

In [ ]:
# 核心要求 2：切分為多個 chunk 模擬串流
N_CHUNKS = 10
chunks = split_into_chunks(X, y, n_chunks=N_CHUNKS)

print(f'總共切分為 {len(chunks)} 個 chunk')
for i, (Xc, yc) in enumerate(chunks):
    print(f'  Chunk {i:2d}: X={Xc.shape}, y={yc.shape}, 類別分布={np.bincount(yc)}')

## Cell 5 — 建立兩條 Pipeline

建立兩個不同的串流 Pipeline 以便後續比較：

- **Random Forest Pipeline**：`StandardScaler` → `RandomForestClassifier`（feature 隨機採樣）
- **Bagging Pipeline**：`StandardScaler` → `EnsembleClassifier(method='bagging')`（全 feature，bootstrap 採樣）

兩個 Pipeline 均透過 `partial_fit` 支援增量學習，不需要看過所有資料才能預測。

In [ ]:
def make_rf_pipeline(seed=42):
    """Random Forest Pipeline：使用 sqrt 特徵採樣，適合高維資料。"""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=10, max_depth=5, random_state=seed
        )),
    ])

def make_bag_pipeline(seed=42):
    """Bagging Pipeline：每棵樹使用全部特徵，以 bootstrap 取樣增加多樣性。"""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', EnsembleClassifier(
            n_estimators=10, method='bagging', max_depth=5, random_state=seed
        )),
    ])

print('Pipeline 建立完成')
print('  RF Pipeline  : StandardScaler → RandomForestClassifier(n=10, depth=5)')
print('  Bag Pipeline : StandardScaler → EnsembleClassifier(bagging, n=10, depth=5)')

## Cell 6 — 以 `StreamTrainer` 進行增量訓練（核心要求 3）

`StreamTrainer` 在每個 chunk 上呼叫 `pipeline.partial_fit(X_chunk, y_chunk)`，
實現真正的增量學習（模型狀態逐步更新，不重置）。

訓練後立即在**同一個 chunk** 上預測，更新 `Accuracy` 和 `F1Score` 的**累積值**（所有 chunk 的預測均計入），
同時記錄每個時間點的記憶體用量（RSS，單位 MB）。

> **指標說明**：`StreamTrainer` 的指標為**累積型**，反映模型在「截至目前所有資料」上的整體表現，
> 而非單一 chunk 的瞬時表現。這能追蹤模型隨資料增加的整體學習進度。

In [ ]:
# 核心要求 3：對每個 chunk 呼叫 .partial_fit() 進行增量訓練

# --- Random Forest 訓練 ---
rf_trainer = StreamTrainer(
    pipeline=make_rf_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)

print('Random Forest 增量訓練中...')
for i, (Xc, yc) in enumerate(chunks):
    record = rf_trainer.fit_chunk(Xc, yc)  # 內部呼叫 pipeline.partial_fit()
    print(f'  Chunk {i:2d} | accuracy={record["accuracy"]:.4f} | f1={record["f1score"]:.4f} | mem={record["memory_mb"]:.1f} MB')

rf_log = rf_trainer.get_log()
rf_acc   = [r['accuracy'] for r in rf_log]
rf_f1    = [r['f1score']  for r in rf_log]
rf_error = [1.0 - a for a in rf_acc]   # error rate = 1 - accuracy

# --- Bagging 訓練 ---
bag_trainer = StreamTrainer(
    pipeline=make_bag_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)

print('\nBagging 增量訓練中...')
for i, (Xc, yc) in enumerate(chunks):
    record = bag_trainer.fit_chunk(Xc, yc)
    print(f'  Chunk {i:2d} | accuracy={record["accuracy"]:.4f} | f1={record["f1score"]:.4f} | mem={record["memory_mb"]:.1f} MB')

bag_log = bag_trainer.get_log()
bag_acc   = [r['accuracy'] for r in bag_log]
bag_f1    = [r['f1score']  for r in bag_log]
bag_error = [1.0 - a for a in bag_acc]

## Cell 7 — 指標趨勢視覺化（核心要求 4）

以 `plot_metric_over_time`（`visualise.py`）繪製 Random Forest 的
**Accuracy** 與 **Error Rate** 隨 chunk 增加的變化趨勢。

預期觀察：隨著訓練資料累積，模型準確率逐漸提升、錯誤率下降，
展示串流學習的核心效益。

In [ ]:
# 核心要求 4（a）：Accuracy 趨勢圖
plot_metric_over_time(
    rf_acc,
    title='Random Forest — Cumulative Accuracy over Chunks',
    ylabel='Accuracy',
    save_path='/tmp/rf_accuracy.png',
)
plt.show()
print(f'Accuracy  : 初始={rf_acc[0]:.4f} → 最終={rf_acc[-1]:.4f}')
print('已儲存 /tmp/rf_accuracy.png')

In [ ]:
# 核心要求 4（b）：Error Rate 趨勢圖
plot_metric_over_time(
    rf_error,
    title='Random Forest — Cumulative Error Rate over Chunks',
    ylabel='Error Rate (1 − Accuracy)',
    save_path='/tmp/rf_error.png',
)
plt.show()
print(f'Error Rate: 初始={rf_error[0]:.4f} → 最終={rf_error[-1]:.4f}')
print('已儲存 /tmp/rf_error.png')

In [ ]:
# 核心要求 4（c）：F1 Score 趨勢圖
plot_metric_over_time(
    rf_f1,
    title='Random Forest — Cumulative F1 Score over Chunks',
    ylabel='F1 Score',
    save_path='/tmp/rf_f1.png',
)
plt.show()
print(f'F1 Score  : 初始={rf_f1[0]:.4f} → 最終={rf_f1[-1]:.4f}')
print('已儲存 /tmp/rf_f1.png')

## Cell 8 — 模型比較視覺化（核心要求 4）

以 `compare_models`（`visualise.py`）在同一張圖上疊加兩個模型的指標曲線，
直觀比較 Random Forest 與 Bagging 的 Accuracy 及 F1 Score 差異。

兩者的主要差別在於特徵採樣策略：Random Forest 每棵樹只使用 √d 個特徵，
Bagging 則使用全部特徵，對本資料集（D=6）差距較小。

In [ ]:
# 核心要求 4（d）：模型 Accuracy 比較
compare_models(
    rf_acc,
    bag_acc,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative Accuracy over Chunks',
    ylabel='Accuracy',
    save_path='/tmp/model_comparison_acc.png',
)
plt.show()
print(f'RF  最終 accuracy : {rf_acc[-1]:.4f}')
print(f'Bag 最終 accuracy : {bag_acc[-1]:.4f}')
print('已儲存 /tmp/model_comparison_acc.png')

In [ ]:
# 核心要求 4（e）：模型 F1 Score 比較
compare_models(
    rf_f1,
    bag_f1,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative F1 Score over Chunks',
    ylabel='F1 Score',
    save_path='/tmp/model_comparison_f1.png',
)
plt.show()
print(f'RF  最終 F1 : {rf_f1[-1]:.4f}')
print(f'Bag 最終 F1 : {bag_f1[-1]:.4f}')
print('已儲存 /tmp/model_comparison_f1.png')

## Cell 9 — 最後一個 Chunk 的預測散點圖（核心要求 4）

以 `plot_predictions_vs_ground_truth`（`visualise.py`）視覺化
Random Forest 在**最後一個 chunk** 上的預測結果 vs. 真實標籤。

圖中每個樣本的 Ground Truth（圓形）與 Prediction（叉形）重疊表示預測正確，
不重疊則為錯誤分類。這有助於快速識別模型在哪些樣本上的表現較差。

In [ ]:
Xlast, ylast = chunks[-1]
y_pred_last = rf_trainer.pipeline.predict(Xlast)

n_correct = int(np.sum(y_pred_last == ylast))
n_total   = len(ylast)

plot_predictions_vs_ground_truth(
    ylast,
    y_pred_last,
    title=f'RF Predictions vs Ground Truth — Last Chunk (chunk {N_CHUNKS-1})',
    save_path='/tmp/pred_vs_true.png',
)
plt.show()
print(f'最後一個 chunk：正確 {n_correct}/{n_total}，chunk accuracy={n_correct/n_total:.4f}')
print('已儲存 /tmp/pred_vs_true.png')

## Cell 10 — 混淆矩陣（核心要求 4）

以 `plot_confusion_matrix`（`visualise.py`）繪製最後一個 chunk 的混淆矩陣熱圖。

混淆矩陣顯示：
- 對角線：預測正確的樣本數（True Positive、True Negative）
- 非對角線：預測錯誤（False Positive、False Negative）

顏色深淺代表數量大小，格內數字為實際計數。

In [ ]:
cm = compute_cm(ylast, y_pred_last)

plot_confusion_matrix(
    cm,
    class_names=['Class 0', 'Class 1'],
    save_path='/tmp/confusion_matrix.png',
)
plt.show()
print('混淆矩陣（最後一個 chunk）:')
print(f'  TN={cm[0,0]:3d}  FP={cm[0,1]:3d}')
print(f'  FN={cm[1,0]:3d}  TP={cm[1,1]:3d}')
print('已儲存 /tmp/confusion_matrix.png')

## Cell 11 — 記憶體用量追蹤（核心要求 4）

以 `plot_metric_over_time`（`visualise.py`）追蹤 Random Forest Pipeline
在串流訓練過程中的 RSS 記憶體用量（單位：MB）。

串流學習的一大優點是記憶體用量應保持穩定，
不隨資料累積量線性增長（不同於批次學習需要儲存全部資料）。

In [ ]:
mem_mb = [r.get('memory_mb', 0.0) for r in rf_log]

plot_metric_over_time(
    mem_mb,
    title='RF Pipeline — Memory Usage (RSS) over Chunks',
    ylabel='RSS Memory (MB)',
    save_path='/tmp/memory_usage.png',
)
plt.show()
print(f'初始記憶體 : {mem_mb[0]:.1f} MB')
print(f'峰值記憶體 : {max(mem_mb):.1f} MB')
print(f'最終記憶體 : {mem_mb[-1]:.1f} MB')
print('已儲存 /tmp/memory_usage.png')

## Cell 12 — 最終結果摘要

彙整兩個模型的最終指標，確認串流訓練的整體效果，
並驗證所有四個核心要求均已滿足。

In [ ]:
print('=' * 50)
print('      串流訓練最終結果摘要')
print('=' * 50)
print(f'  資料集     : {N} 筆 × {D} 特徵，二元分類')
print(f'  Chunk 數   : {N_CHUNKS} 個（每個 {N // N_CHUNKS} 筆）')
print()
header = f"  {'模型':<20} {'Accuracy':>10} {'F1 Score':>10}"
print(header)
print(f'  {"-"*42}')
print(f'  {"Random Forest":<20} {rf_acc[-1]:>10.4f} {rf_f1[-1]:>10.4f}')
print(f'  {"Bagging":<20} {bag_acc[-1]:>10.4f} {bag_f1[-1]:>10.4f}')
print()
print('  核心要求確認：')
print('  1. 使用 io.load_csv 從 CSV 載入資料集')
print('  2. 使用 split_into_chunks 切分為 10 個 chunk（模擬串流）')
print('  3. 對每個 chunk 呼叫 pipeline.partial_fit() 進行增量訓練')
print('  4. 以 visualise.py 記錄並視覺化：')
print('       - Accuracy 趨勢    (/tmp/rf_accuracy.png)')
print('       - Error Rate 趨勢  (/tmp/rf_error.png)')
print('       - F1 Score 趨勢    (/tmp/rf_f1.png)')
print('       - 模型比較 (Acc)   (/tmp/model_comparison_acc.png)')
print('       - 模型比較 (F1)    (/tmp/model_comparison_f1.png)')
print('       - 預測散點圖       (/tmp/pred_vs_true.png)')
print('       - 混淆矩陣         (/tmp/confusion_matrix.png)')
print('       - 記憶體用量       (/tmp/memory_usage.png)')